# Notebook 08 — XGBoost

Algoritmo: XGBoost. Escalado: no. n_iter = 50, CV=5, scoring='roc_auc'.

Observaciones: Gradient boosting SOTA tabular.

## 1. Setup

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

RAIZ = Path('..').resolve()
sys.path.insert(0, str(RAIZ))

from src.preprocesamiento import preparar, RANDOM_STATE
from src.entrenamiento  import entrenar_y_evaluar

DATA = RAIZ / 'data'

from xgboost import XGBClassifier
from scipy.stats import loguniform, uniform

## 2. Cargar y preparar datos

In [2]:
datos = preparar(DATA, aplicar_escalado=False)
print(f'Train: {datos.n_train} | Test: {datos.n_test} | Features: {len(datos.cols_features)}')
print(f'scale_pos_weight atraso: {datos.scale_pos_weight["tuvo_atraso"]:.4f}')


Train: 38664 | Test: 9667 | Features: 152
scale_pos_weight atraso: 0.6534


## 3. Clasificador + grid

In [3]:
clf = XGBClassifier(objective='binary:logistic', eval_metric='logloss',
    tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE,
    scale_pos_weight=datos.scale_pos_weight['tuvo_atraso'])

param_distributions = {
        'n_estimators': [300, 500, 800],
        'learning_rate': loguniform(0.01, 0.3),
        'max_depth': [6, 8, 10, 12],
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4),
        'gamma': uniform(0, 5),
        'reg_lambda': loguniform(0.1, 10),
        'reg_alpha': loguniform(0.001, 1.0),
    }

## 4. RandomizedSearchCV (50 iters × CV 5)

In [4]:
resultado = entrenar_y_evaluar(
    nombre='xgb',
    estimador=clf,
    param_distributions=param_distributions,
    datos=datos,
    data_dir=DATA,
    n_iter=50,
    target='tuvo_atraso',
)
resultado


=== XGB ===
CV AUC:          0.9085
Test AUC:        0.9069
Test F1:         0.85
Test Precision:  0.8765
Test Recall:     0.825
Test Accuracy:   0.8238
Tiempo search:   185.7 s
Best params:     {'colsample_bytree': 0.8886918084659492, 'gamma': 1.1799245987447788, 'learning_rate': 0.02389152997793864, 'max_depth': 10, 'n_estimators': 300, 'reg_alpha': 0.1355154055297656, 'reg_lambda': 0.16664091501238187, 'subsample': 0.7757346007463081}


{'nombre': 'xgb',
 'target': 'tuvo_atraso',
 'n_iter': 50,
 'cv_folds': 5,
 'tiempo_search_seg': 185.7,
 'best_params': {'colsample_bytree': 0.8886918084659492,
  'gamma': 1.1799245987447788,
  'learning_rate': 0.02389152997793864,
  'max_depth': 10,
  'n_estimators': 300,
  'reg_alpha': 0.1355154055297656,
  'reg_lambda': 0.16664091501238187,
  'subsample': 0.7757346007463081},
 'best_cv_auc': 0.9085,
 'AUC': 0.9069,
 'F1': 0.85,
 'Precision': 0.8765,
 'Recall': 0.825,
 'Accuracy': 0.8238}